# 24c — Yorkshire SDP Case Study Validation v1

This notebook focuses on Yorkshire and neighbouring SDP case-study areas to test the model against actual SDP campaign evidence.

Core questions:

- What do SDP-contested Yorkshire wards look like demographically?
- Which tribes appear in high-performing SDP wards?
- Do successful SDP cases resemble North West watchlist wards?
- Does Middleton Park / Leeds / South Yorkshire evidence support or challenge the current tribe interpretation?

Outputs are written to:

```text
data/processed/yorkshire_case_study_v1/
```

In [2]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

INPUT_DIRS = [
    PROCESSED_DIR / "sdp_campaign_validation_v2",
    PROCESSED_DIR / "sdp_validation_v2",
    PROCESSED_DIR / "target_review_pack_v1",
    PROCESSED_DIR / "caveat_resolution_v2",
    PROCESSED_DIR / "target_model_v2",
    PROCESSED_DIR,
]

OUTPUT_DIR = PROCESSED_DIR / "yorkshire_case_study_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SDP_PROFILE_FILENAME = "sdp_campaign_wards_profile_v2.csv"
NW_REVIEW_FILENAME = "north_west_revised_consolidated_review_v2.csv"

YORKSHIRE_REGION_NAMES = ["Yorkshire and The Humber", "Yorkshire", "Yorkshire & The Humber"]
KEY_CASE_STUDY_LADS = [
    "Leeds", "Barnsley", "Doncaster", "Rotherham", "Sheffield", "Wakefield",
    "Kirklees", "Calderdale", "Bradford", "York", "North Yorkshire", "East Riding of Yorkshire", "Kingston upon Hull, City of"
]
SOUTH_YORKSHIRE_LADS = ["Barnsley", "Doncaster", "Rotherham", "Sheffield"]
KEY_WARD_NAMES = ["Middleton Park", "Dearne South", "Wath", "Beeston and Holbeck", "Firth Park"]

print("Project:", PROJECT_DIR)
print("Output:", OUTPUT_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Output: c:\Users\keena\Documents\Electoral_Tribes\data\processed\yorkshire_case_study_v1


In [3]:
def find_file(filename, required=True):
    for folder in INPUT_DIRS + [Path("/mnt/data")]:
        p = folder / filename
        if p.exists():
            return p
    for root in [PROCESSED_DIR, PROJECT_DIR]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return sorted(matches, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    if required:
        raise FileNotFoundError(filename)
    return None


def clean_text(x):
    if pd.isna(x): return ""
    import re
    x = str(x).lower().strip().replace("&", " and ")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def read_csv(filename, required=True):
    p = find_file(filename, required=required)
    if p is None:
        return None
    df = pd.read_csv(p, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {p}")
    return df

## 24c.1 Load SDP validation file and identify Yorkshire case-study rows

In [4]:

sdp = read_csv(SDP_PROFILE_FILENAME)

# ---------------------------------------------------------------------
# Robust field coalescing
# ---------------------------------------------------------------------
# Some SDP validation rows have raw council/ward names, some have model-
# matched LAD25/WD25 names, and some have both. Use the model names where
# available, then fall back to the raw source names.

def coalesce_columns(df, columns, default=""):
    out = pd.Series(default, index=df.index, dtype="object")
    for col in columns:
        if col in df.columns:
            candidate = df[col]
            candidate = candidate.where(candidate.notna(), "")
            candidate = candidate.astype(str).str.strip()
            out = out.where(out.astype(str).str.strip().ne(""), candidate)
    return out.fillna("")

sdp["analysis_region"] = coalesce_columns(sdp, ["analysis_region", "analysis_region_model"])
sdp["LAD25NM_display"] = coalesce_columns(sdp, ["LAD25NM", "LAD25NM_model", "council_name"])
sdp["WD25NM_display"] = coalesce_columns(sdp, ["WD25NM", "WD25NM_model", "ward_name"])
sdp["LAD25CD_display"] = coalesce_columns(sdp, ["LAD25CD", "LAD25CD_model"])
sdp["WD25CD_display"] = coalesce_columns(sdp, ["WD25CD", "WD25CD_final", "WD25CD_model", "ward_code"])

# Preserve expected column names for later cells, but use display values.
sdp["LAD25NM"] = sdp["LAD25NM_display"]
sdp["WD25NM"] = sdp["WD25NM_display"]
sdp["LAD25CD"] = sdp["LAD25CD_display"]
sdp["WD25CD"] = sdp["WD25CD_display"]

sdp["clean_region"] = sdp["analysis_region"].map(clean_text)
sdp["clean_lad"] = sdp["LAD25NM"].map(clean_text)
sdp["clean_ward"] = sdp["WD25NM"].map(clean_text)

case_lads_clean = {clean_text(x) for x in KEY_CASE_STUDY_LADS}
south_yorks_clean = {clean_text(x) for x in SOUTH_YORKSHIRE_LADS}
key_wards_clean = {clean_text(x) for x in KEY_WARD_NAMES}
yorkshire_region_clean = {clean_text(x) for x in YORKSHIRE_REGION_NAMES}

# Use cleaned values so that Yorkshire & The Humber / Yorkshire and The Humber
# variants do not cause a false miss.
yorkshire_mask = (
    sdp["clean_region"].isin(yorkshire_region_clean)
    | sdp["clean_lad"].isin(case_lads_clean)
    | sdp["clean_ward"].isin(key_wards_clean)
)

yorks = sdp.loc[yorkshire_mask].copy()

# ---------------------------------------------------------------------
# Case-study labels, vectorised rather than apply(axis=1)
# ---------------------------------------------------------------------
# This avoids a pandas edge case where apply on an empty dataframe can return
# a dataframe rather than a series, causing assignment errors.
conditions = [
    yorks["clean_ward"].str.contains("middleton park", na=False),
    yorks["clean_ward"].str.contains("beeston", na=False) & yorks["clean_ward"].str.contains("holbeck", na=False),
    yorks["clean_lad"].eq(clean_text("Leeds")),
    yorks["clean_lad"].eq(clean_text("Barnsley")),
    yorks["clean_lad"].eq(clean_text("Doncaster")),
    yorks["clean_lad"].eq(clean_text("Rotherham")),
    yorks["clean_lad"].eq(clean_text("Sheffield")),
    yorks["clean_lad"].isin(south_yorks_clean),
]
choices = [
    "Leeds - Middleton Park",
    "Leeds - Beeston and Holbeck",
    "Leeds - Other",
    "Barnsley",
    "Doncaster",
    "Rotherham",
    "Sheffield",
    "South Yorkshire - Other",
]
yorks["case_study_area"] = np.select(conditions, choices, default="Other Yorkshire / Humber")

print("Yorkshire / case-study SDP rows:", len(yorks))

if len(yorks) == 0:
    print("No Yorkshire/case-study rows were selected. Check analysis_region, LAD25NM and WD25NM values below:")
    display(sdp[["analysis_region", "LAD25NM", "WD25NM", "council_name", "ward_name"]].head(30))
    raise ValueError("No Yorkshire/case-study SDP rows selected. Input file or region/council names need review.")

summary_area = yorks["case_study_area"].value_counts(dropna=False).reset_index()
summary_area.columns = ["case_study_area", "rows"]
display(summary_area)


Loaded sdp_campaign_wards_profile_v2.csv: (171, 64) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\sdp_campaign_validation_v2\sdp_campaign_wards_profile_v2.csv
Yorkshire / case-study SDP rows: 100


,case_study_area,rows
0,Leeds - Other,50
1,Other Yorkshire / Humber,19
2,Barnsley,12
3,Rotherham,7
4,Sheffield,4
5,Leeds - Middleton Park,4
6,Leeds - Beeston and Holbeck,3
7,Doncaster,1


## 24c.2 Produce case-study summaries

In [5]:
def summarise(group_cols, filename):
    cols = [c for c in group_cols if c in yorks.columns]
    if not cols:
        return pd.DataFrame()
    out = (
        yorks.groupby(cols, dropna=False, as_index=False)
        .agg(
            contests=("candidate_name", "size"),
            years=("election_year", lambda s: "; ".join(map(str, sorted(set(s.dropna().astype(int)))) if s.notna().any() else "")),
            total_sdp_votes=("sdp_votes", "sum"),
            mean_sdp_vote_share=("sdp_vote_share_effective", "mean"),
            median_sdp_vote_share=("sdp_vote_share_effective", "median"),
            max_sdp_vote_share=("sdp_vote_share_effective", "max"),
            mean_model_score=("initial_watchlist_score", "mean"),
            max_model_score=("initial_watchlist_score", "max"),
        )
        .sort_values(["max_sdp_vote_share", "total_sdp_votes"], ascending=False)
    )
    out.to_csv(OUTPUT_DIR / filename, index=False)
    return out

case_wards = summarise(["case_study_area", "LAD25NM", "WD25NM", "WD25CD", "dominant_cluster_name", "second_cluster_name", "latest_election_top_party_bucket"], "yorkshire_sdp_case_study_wards_v1.csv")
by_tribe = summarise(["dominant_cluster_name"], "yorkshire_sdp_performance_by_tribe_v1.csv")
by_party = summarise(["latest_election_top_party_bucket"], "yorkshire_sdp_performance_by_latest_party_v1.csv")
by_area = summarise(["case_study_area"], "yorkshire_sdp_performance_by_case_area_v1.csv")

# Specific Middleton Park profile.
middleton = yorks[yorks["case_study_area"].eq("Leeds - Middleton Park")].copy()
middleton.to_csv(OUTPUT_DIR / "middleton_park_case_study_profile_v1.csv", index=False)

# High performance cases.
high_perf = yorks.sort_values(["sdp_vote_share_effective", "sdp_votes"], ascending=False).head(50).copy()
high_perf.to_csv(OUTPUT_DIR / "yorkshire_high_performance_sdp_cases_v1.csv", index=False)

print("Case ward summary:")
display(case_wards.head(20))
print("Performance by tribe:")
display(by_tribe)
print("Middleton Park rows:", len(middleton))
display(middleton.head(10))

Case ward summary:


,case_study_area,LAD25NM,WD25NM,WD25CD,dominant_cluster_name,second_cluster_name,latest_election_top_party_bucket,contests,years,total_sdp_votes,mean_sdp_vote_share,median_sdp_vote_share,max_sdp_vote_share,mean_model_score,max_model_score
8,Leeds - Middleton Park,Leeds,Middleton Park,E05011404,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,sdp,4,2021; 2022; 2023; 2024,8517,0.436519,0.444429,0.507556,68.685051,68.685051
2,Barnsley,Barnsley,Dearne South,E05000982,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,lab,4,2021; 2022; 2023; 2024,1316,0.173728,0.191558,0.248884,57.831631,57.831631
49,Rotherham,Rotherham,Wath,E05013016,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,lab,3,2021; 2024,716,0.136329,0.144279,0.203901,65.710858,65.710858
0,Barnsley,Barnsley,Darfield,E05000978,Settled Working Families / Skilled Trades Suburbs,Post-Industrial Estates / Deprived Working Com...,lab,1,2022,208,0.105637,0.105637,0.105637,60.253467,60.253467
36,Other Yorkshire / Humber,East Riding of Yorkshire,Bridlington Central and Old Town,E05001688,Post-Industrial Estates / Deprived Working Com...,Rooted Older Homeowners,independent,1,2023,228,0.086070,0.086070,0.086070,73.084451,73.084451
51,Sheffield,Sheffield,Firth Park,E05010868,Post-Industrial Estates / Deprived Working Com...,Settled Diverse Urban Communities,lab,2,2022; 2023,438,0.074460,0.074460,0.083470,62.455189,62.455189
7,Leeds - Beeston and Holbeck,Leeds,Beeston & Holbeck,E05012647,Settled Diverse Urban Communities,Settled Working Families / Skilled Trades Suburbs,lab,3,2022; 2023; 2024,604,0.045333,0.051641,0.057379,44.412659,44.412659
52,Sheffield,Sheffield,Park and Arbourthorne,E05010876,Post-Industrial Estates / Deprived Working Com...,Rooted Older Homeowners,lab,1,2024,175,0.055362,0.055362,0.055362,53.292779,53.292779
47,Rotherham,Rotherham,Kilnhurst & Swinton East,E05013005,Rooted Older Homeowners,Settled Working Families / Skilled Trades Suburbs,lab,2,2021,187,0.048072,0.048072,0.052956,58.579775,58.579775
5,Barnsley,Barnsley,Wombwell,E05000995,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,lab,2,2021; 2022,111,0.026151,0.026151,0.045301,53.395980,53.395980


Performance by tribe:


,dominant_cluster_name,contests,years,total_sdp_votes,mean_sdp_vote_share,median_sdp_vote_share,max_sdp_vote_share,mean_model_score,max_model_score
0,Post-Industrial Estates / Deprived Working Com...,37,2021; 2022; 2023; 2024,12391,0.093819,0.025355,0.507556,58.685930,73.773860
3,Settled Working Families / Skilled Trades Suburbs,21,2021; 2022; 2023; 2024,1124,0.014670,0.010978,0.105637,58.346757,65.631208
2,Settled Diverse Urban Communities,13,2021; 2022; 2023; 2024,1215,0.019595,0.011338,0.057379,39.509142,44.852798
1,Rooted Older Homeowners,15,2021; 2022; 2023; 2024,829,0.019641,0.012086,0.052956,54.934553,61.596964
4,Stable Suburban Professionals,13,2021; 2022; 2023; 2024,758,0.007776,0.007073,0.018087,41.510204,50.835353
5,Student & Transient Youth,1,2021,10,0.001843,0.001843,0.001843,36.753970,36.753970


Middleton Park rows: 4


,election_year,election_date,council_name,ward_name,candidate_name,party_label,sdp_votes,valid_votes,vote_share,sdp_position,winner_party,runner_up_party,WD25CD,WD25NM,LAD25CD,LAD25NM,ward_code,result_area_key,boundary_year,source_mode,source_file,source_sheet,source_url,source_notes,mapping_confidence,mapping_notes,result_id,source_year,sdp_vote_share_raw_clean,sdp_vote_share_from_votes,sdp_vote_share_effective,clean_council_name,clean_ward_name,patched_source_geography_code,patched_source_geography_name,patched_ward_code,patched_lad_code,patched_source_boundary_year,source_geography_code_for_match,source_boundary_year_for_match,match_method_v2,match_confidence_v2,WD25CD_final,wd25_match_share_v2,source_boundary_year,source_ward_code,crosswalk_WD25CD,wd25_match_share,WD25CD_model,WD25NM_model,LAD25CD_model,LAD25NM_model,analysis_region,strategic_lane,initial_watchlist_score,demographic_relevance_score,electoral_opportunity_score,political_openness_score,breakthrough_complacency_score,dominant_cluster_name,second_cluster_name,latest_election_top_party_bucket,latest_election_runner_up_party_bucket,matched_to_model_v2,LAD25NM_display,WD25NM_display,LAD25CD_display,WD25CD_display,clean_region,clean_lad,clean_ward,case_study_area
59,2021,2021-05-06,Leeds,Middleton Park,Dixon W.A.,SDP,1963,5614.0,0.349662,NaN,NaN,NaN,E05011404,Middleton Park,E08000035,Leeds,E05011404,2021|CODE|LEEDS|E05011404|MIDDLETON_PARK,2021,existing_sdp_profile_v1,local_elections_2021_results-2.xlsx,Candidates-results,NaN,House of Commons Library Local Election Handbo...,NaN,NaN,459fa8e908df8518,2021,0.349662,0.349662,0.349662,leeds,middleton park,NaN,NaN,E05011404,E08000035,NaN,E05011404,2021,existing_wd25cd,high,E05011404,NaN,NaN,NaN,NaN,NaN,E05011404,Middleton Park,E08000035,Leeds,Yorkshire and The Humber,Clean Opportunity,68.685051,80.823173,59.041825,50.737571,0.0,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,sdp,lab,True,Leeds,Middleton Park,E08000035,E05011404,yorkshire and the humber,leeds,middleton park,Leeds - Middleton Park
93,2022,2022-05-05,Leeds,Middleton Park,Dixon W.,SDP,2687,5294.0,0.507556,NaN,NaN,NaN,E05011404,Middleton Park,E08000035,Leeds,E05011404,2022|CODE|LEEDS|E05011404|MIDDLETON_PARK,2022,existing_sdp_profile_v1,local-elections-2022.xlsx,Candidates-results,NaN,House of Commons Library Local Election Handbo...,NaN,NaN,526be0caba99bc6a,2022,0.507556,0.507556,0.507556,leeds,middleton park,NaN,NaN,E05011404,E08000035,NaN,E05011404,2022,existing_wd25cd,high,E05011404,NaN,2022.0,E05011404,E05011404,1.0,E05011404,Middleton Park,E08000035,Leeds,Yorkshire and The Humber,Clean Opportunity,68.685051,80.823173,59.041825,50.737571,0.0,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,sdp,lab,True,Leeds,Middleton Park,E08000035,E05011404,yorkshire and the humber,leeds,middleton park,Leeds - Middleton Park
121,2023,2023-05-04,Leeds,Middleton Park,"Pogson-Golden, E.",SDP,1985,4311.0,0.460450,NaN,NaN,NaN,E05011404,Middleton Park,E08000035,Leeds,NaN,2023|NAME|LEEDS|MIDDLETON_PARK,2023,existing_sdp_profile_v1,LEH-Candidates-2023.xlsx,Cand_Table,NaN,House of Commons Library Local Election Handbo...,NaN,NaN,1b201a737e20c06f,2023,0.460450,0.460450,0.460450,leeds,middleton park,E05011404,Middleton Park,E05011404,E08000035,2023.0,E05011404,2023,source_year_to_wd25_crosswalk,high,E05011404,1.0,2023.0,E05011404,E05011404,1.0,E05011404,Middleton Park,E08000035,Leeds,Yorkshire and The Humber,Clean Opportunity,68.685051,80.823173,59.041825,50.737571,0.0,Post-Industrial Estates / Deprived Working Com...,Settled Working Families / Skilled Trades Suburbs,sdp,lab,True,Leeds,Middleton Park,E08000035,E05011404,yorkshire and the humber,leeds,middleton park,Leeds - Middleton Park
143,2024,2024-05-02,Leeds,Middleton Park,"Chesterfield, R.",SDP,1882,4393.0,0.428409,NaN,NaN,NaN,E05011404,Middleton Park,E08000035,Leeds,E05011404,2024|CODE|LEEDS|E05011404|MIDDLETON_PARK,202

## 24c.3 Compare Yorkshire SDP evidence with North West watchlist profile

This comparison is indicative only. It checks whether high-performing SDP wards resemble the current North West model's preferred lanes and tribes.

In [6]:
# Load NW revised review if available for broad comparison.
nw = read_csv(NW_REVIEW_FILENAME, required=False)
comparison_rows = []

if nw is not None:
    # Tribe comparison.
    y_tribe = yorks["dominant_cluster_name"].value_counts(normalize=True).rename("yorkshire_sdp_share")
    nw_tribe = nw["dominant_cluster_name"].value_counts(normalize=True).rename("north_west_review_share")
    tribe_compare = pd.concat([y_tribe, nw_tribe], axis=1).fillna(0).reset_index().rename(columns={"index": "dominant_cluster_name"})
    tribe_compare["difference_yorkshire_minus_nw"] = tribe_compare["yorkshire_sdp_share"] - tribe_compare["north_west_review_share"]
    tribe_compare.to_csv(OUTPUT_DIR / "yorkshire_vs_north_west_tribe_comparison_v1.csv", index=False)
    display(tribe_compare.sort_values("difference_yorkshire_minus_nw", ascending=False))

    # Latest top party comparison.
    if "latest_election_top_party_bucket" in yorks.columns and "latest_election_top_party_bucket" in nw.columns:
        y_party = yorks["latest_election_top_party_bucket"].value_counts(normalize=True).rename("yorkshire_sdp_share")
        nw_party = nw["latest_election_top_party_bucket"].value_counts(normalize=True).rename("north_west_review_share")
        party_compare = pd.concat([y_party, nw_party], axis=1).fillna(0).reset_index().rename(columns={"index": "latest_election_top_party_bucket"})
        party_compare["difference_yorkshire_minus_nw"] = party_compare["yorkshire_sdp_share"] - party_compare["north_west_review_share"]
        party_compare.to_csv(OUTPUT_DIR / "yorkshire_vs_north_west_latest_party_comparison_v1.csv", index=False)
        display(party_compare.sort_values("difference_yorkshire_minus_nw", ascending=False))

# Case-study headline summary.
summary = pd.DataFrame([{
    "case_study_rows": len(yorks),
    "matched_to_model_rows": int(yorks.get("matched_to_model_v2", pd.Series(False)).fillna(False).astype(bool).sum()) if "matched_to_model_v2" in yorks.columns else int(yorks["initial_watchlist_score"].notna().sum()),
    "middleton_park_rows": len(middleton),
    "max_sdp_vote_share": yorks["sdp_vote_share_effective"].max(),
    "mean_sdp_vote_share": yorks["sdp_vote_share_effective"].mean(),
    "top_dominant_tribe": yorks["dominant_cluster_name"].mode().iloc[0] if yorks["dominant_cluster_name"].notna().any() else "unknown",
    "top_latest_party": yorks["latest_election_top_party_bucket"].mode().iloc[0] if yorks["latest_election_top_party_bucket"].notna().any() else "unknown",
}])
summary.to_csv(OUTPUT_DIR / "yorkshire_case_study_summary_v1.csv", index=False)
display(summary)

Loaded north_west_revised_consolidated_review_v2.csv: (825, 74) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\caveat_resolution_v2\north_west_revised_consolidated_review_v2.csv


,dominant_cluster_name,yorkshire_sdp_share,north_west_review_share,difference_yorkshire_minus_nw
0,Post-Industrial Estates / Deprived Working Com...,0.37,0.201212,0.168788
4,Settled Diverse Urban Communities,0.13,0.070303,0.059697
1,Settled Working Families / Skilled Trades Suburbs,0.21,0.201212,0.008788
6,Cosmopolitan Young Professional Core,0.00,0.010909,-0.010909
5,Student & Transient Youth,0.01,0.026667,-0.016667
3,Stable Suburban Professionals,0.13,0.207273,-0.077273
2,Rooted Older Homeowners,0.15,0.282424,-0.132424


,latest_election_top_party_bucket,yorkshire_sdp_share,north_west_review_share,difference_yorkshire_minus_nw
0,lab,0.67,0.490909,0.179091
4,sdp,0.04,0.000000,0.040000
2,other,0.07,0.033939,0.036061
3,green,0.06,0.041212,0.018788
1,ld,0.09,0.106667,-0.016667
6,independent,0.02,0.069091,-0.049091
5,con,0.04,0.139394,-0.099394
7,reform_ukip_brexit,0.01,0.118788,-0.108788


,case_study_rows,matched_to_model_rows,middleton_park_rows,max_sdp_vote_share,mean_sdp_vote_share,top_dominant_tribe,top_latest_party
0,100,100,4,0.507556,0.044317,Post-Industrial Estates / Deprived Working Com...,lab
